In [1]:
# Cài đặt công cụ hệ thống để chuyển PDF sang ảnh
!apt-get install -y poppler-utils

# Cài đặt các thư viện Python cần thiết
!pip install -q transformers qwen-vl-utils pdf2image accelerate torch torchvision

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 2s (95.1 kB/s)                    
Selecting previously unselected package poppler-utils.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 64.4 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
import os
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from pdf2image import convert_from_path
from PIL import Image

class AudioBookPDFParser:
    def __init__(self):
        print("Đang tải mô hình Qwen2.5-VL-3B-Instruct lên GPU T4...")
        # Sử dụng float16 tối ưu cho GPU T4
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            "Qwen/Qwen2.5-VL-3B-Instruct",
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
        print("Tải mô hình thành công!")

    def infer(self, pdf_path: str, output_mmd_path: str, dpi: int = 200):
        """
        Hàm xử lý PDF. Lưu ý: hàm này không trả ra string mà kết quả 
        được append trực tiếp vào file mmd để phục vụ pipeline tiếp theo.
        """
        print(f"Bắt đầu chuyển đổi {pdf_path} sang hình ảnh...")
        # Chuyển PDF thành list các ảnh PIL. DPI=200 là đủ nét cho OCR và tiết kiệm RAM
        images = convert_from_path(pdf_path, dpi=dpi)
        
        # Đảm bảo thư mục chứa file đầu ra tồn tại
        os.makedirs(os.path.dirname(output_mmd_path) if os.path.dirname(output_mmd_path) else ".", exist_ok=True)

        print(f"Bắt đầu phân tích cấu trúc và ghi vào: {output_mmd_path}")
        # Mở file .mmd ở chế độ ghi (overwrite nếu đã có)
        with open(output_mmd_path, "w", encoding="utf-8") as mmd_file:
            for page_num, img in enumerate(images):
                print(f"Đang xử lý trang {page_num + 1}/{len(images)}...")
                
                # Prompt được tinh chỉnh chuyên biệt cho đầu vào TTS
                prompt_text = (
                    "Hãy trích xuất toàn bộ văn bản trong hình ảnh tài liệu này theo đúng thứ tự đọc. "
                    "Sử dụng định dạng Markdown (# cho tiêu đề chính, ## cho tiêu đề phụ). "
                    "TUYỆT ĐỐI BỎ QUA các thông tin nhiễu như: số trang, tiêu đề đầu trang (header), "
                    "chân trang (footer), và các ghi chú hình ảnh để luồng đọc âm thanh không bị vấp."
                )

                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": prompt_text},
                        ],
                    }
                ]

                # Chuẩn bị input cho mô hình
                text = self.processor.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                image_inputs, video_inputs = process_vision_info(messages)
                inputs = self.processor(
                    text=[text],
                    images=image_inputs,
                    videos=video_inputs,
                    padding=True,
                    return_tensors="pt",
                ).to("cuda")

                # Sinh text (điều chỉnh max_new_tokens nếu trang PDF có chữ rất dày đặc)
                generated_ids = self.model.generate(**inputs, max_new_tokens=1500)
                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
                
                output_text = self.processor.batch_decode(
                    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
                )[0]

                # Ghi text của trang hiện tại vào file mmd, thêm khoảng trắng giữa các trang
                mmd_file.write(f"\n")
                mmd_file.write(output_text.strip() + "\n\n")

        print("Hoàn tất xử lý PDF!")

# --- Cách sử dụng ---
# Giả sử bạn đã upload một file 'sach_mau.pdf' lên Colab


In [ ]:
from pathlib import Path

def select_pdf_file():
    """Chọn file PDF trong VS Code/Jupyter theo cách ổn định."""
    # 1) Ưu tiên hộp thoại chọn file local
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        selected = filedialog.askopenfilename(
            title="Chọn file PDF",
            filetypes=[("PDF files", "*.pdf"), ("All files", "*.*")]
        )
        root.destroy()
        if selected:
            return selected
    except Exception:
        pass

    # 2) Fallback: nhập đường dẫn thủ công
    manual = input("Nhập đường dẫn file PDF: ").strip().strip('"').strip("'")
    if manual:
        return manual

    raise RuntimeError("Chưa chọn được file PDF.")

pdf_file = select_pdf_file()
if not Path(pdf_file).exists():
    raise FileNotFoundError(f"Không tìm thấy file: {pdf_file}")

print(f"Đã chọn file: {pdf_file}")

KeyboardInterrupt: 

In [ ]:
from pathlib import Path

parser = AudioBookPDFParser()
output_name = Path(pdf_file).stem + "_audio.mmd"
parser.infer(pdf_path=pdf_file, output_mmd_path=output_name)
print(f"File kết quả: {output_name}")